# petri clash build walkthrough

this notebook is a full walkthrough of how the project is built, why each file exists, what the code is doing, and what i was thinking while putting it together.

this is not a polished research note. it is closer to a very long engineering brain dump. the point is to explain the real implementation as it exists in this repo, including the mistakes, the fixes, and the tradeoffs.

when i build something like this, i usually split my thinking into four layers:

1. what is the smallest version that is actually the same idea.
2. what parts need to be faithful to the original nca paper.
3. what parts can be simplified so the whole thing stays understandable.
4. what parts are likely to break once two models start sharing space.

petri clash ended up following exactly that path.

## repo shape

before touching details, this is the mental map of the repo.

- `nca.py` is the core learned cellular automaton rule.
- `train.py` trains one nca against one target image.
- `clash.py` loads two trained ncas, places them on one board, and handles the fight.
- `targets/` contains the little icon images.
- `weights/` contains saved checkpoints.

that split is deliberate. i did not want the game loop to know anything about optimization internals, and i did not want the trainer to know anything about pygame.

## the big design idea

the whole project only works if one fact stays true:

an nca is not drawing pixels directly in the way a normal image model would. it is learning a local update law.

that means each cell only gets a tiny local view, updates its own 16-channel state, and then repeats this over and over. so the model is really learning a dynamical system, not a one-shot renderer.

that is why the project feels alive when it works. the same learned rule can:

- grow from a seed
- repair damage
- continue evolving after partial destruction
- compete at a frontier

that also means failure modes are dynamical failure modes. if the update rule gets pushed outside what it saw during training, it does not fail gracefully. it starts inventing weird junk, drifting, leaving sparks behind, and feeding on those mistakes.

## file 1: `nca.py`

this file is the heart of everything.

when i started building it, i tried to keep the model as close as possible to the 2020 distill setup while still staying small enough to understand at a glance.

the choices here are:

- 16 channels per cell
- rgba in the first 4 channels
- 12 hidden channels after that
- depthwise perception using identity, sobel-x, and sobel-y
- a tiny per-cell mlp implemented as two 1x1 convolutions
- stochastic updates with a fire rate of 0.5
- alive masking based on alpha

that is almost the whole nca idea in one page of code.

```python
# --- perception: fixed sobel filters, no learned weights ---

def perceive(self, state):
    # 3x3 sobel_x and sobel_y, applied per-channel
    perception = torch.cat([state, sobel_x_conv, sobel_y_conv], dim=1)
    return perception  # [B, channels*3, H, W]

# --- update: two conv layers, that's the whole brain ---

def update(self, perception):
    out = self.fc0(perception)      # 1x1 conv, channels*3 -> hidden_size
    out = torch.relu(out)
    out = self.fc1(out)             # 1x1 conv, hidden_size -> channels
    return out                       # residual delta, added to state

# --- stochastic firing: not every cell updates every step ---

fire_mask = (torch.rand(...) < self.fire_rate).float()
state = state + delta * fire_mask

# --- alive masking: alpha decides life and death ---

alive = (F.max_pool2d(state[:, 3:4], 3, stride=1, padding=1) > 0.1)
state = state * alive.float()
```

the full source is in `nca.py` if you want to read it — about 90 lines total.

## why the state is 16 channels

four visible channels are not enough.

if i only let each cell keep rgba, the model has almost no working memory. it can paint, but it cannot carry enough latent structure to build shape, repair structure, or negotiate boundaries in an interesting way.

the hidden channels are the model's private scratch space. they are where the internal wavefront, orientation cues, and whatever other latent bookkeeping the model invents can live.

that is why the hidden state is not optional decoration. it is what makes the organism behavior possible.

## why sobel filters show up here

the perception stage is fixed, not learned.

that sounds restrictive at first, but it is actually helpful. each channel gets:

- its own center value through the identity filter
- horizontal differences through sobel-x
- vertical differences through sobel-y

so each cell can cheaply sense both "what am i" and "how is the local pattern changing around me".

using fixed filters keeps the parameter count tiny and makes the system closer to the distill recipe. it also means the learning burden stays in the update rule rather than in a larger perception stack.

In [ ]:
import torch
from nca import NCA

model = NCA()
print('perception filter shape:', tuple(model.perception_filters.shape))
print('fc0 weight shape:', tuple(model.fc0.weight.shape))
print('fc1 weight shape:', tuple(model.fc1.weight.shape))

param_count = sum(p.numel() for p in model.parameters())
print(f'parameter count: {param_count}')
print(f'model size: {sum(p.numel() * p.element_size() for p in model.parameters()) / 1024:.1f} KB')

## why the last layer starts at zero

this is one of those tiny details that matters more than it looks.

if the last layer starts random, then the very first rollout from the seed can be chaotic. the whole grid can explode before learning finds any stable attractor.

if the last layer starts at zero, the model initially behaves like "do almost nothing". then training gradually learns useful residual updates.

that gives a much calmer optimization path.

## the alive mask

this is the line that decides whether dead regions really stay dead.

alpha is treated as the cell's visible aliveness signal. if alpha is weak in a local 3x3 area, the whole cell state gets zeroed out.

there are two reasons for this:

- it prevents invisible hidden junk from surviving forever after visible structure dies
- it keeps the growth front compact instead of letting latent noise linger in dead space

this becomes even more important once damage craters and collisions are involved.

In [ ]:
from nca import make_seed

state = make_seed(1, height=48, width=48, device='cpu')
print('state shape:', tuple(state.shape))
print(f'seed alpha sum: {float(state[:, 3].sum()):.1f}  (one pixel alive)')
print(f'seed hidden channels: all zero — the model starts from nothing')

### let's watch it grow

run a single model forward from its seed and see what happens at each stage.

In [ ]:
import matplotlib.pyplot as plt
from nca import NCA, make_seed
import torch

model = NCA()
checkpoint = torch.load('checkpoints/01_heart.pt', map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint['model'])
model.eval()

state = make_seed(1, height=48, width=48, device='cpu')
snapshots = []

with torch.inference_mode():
    for step in range(1, 201):
        state = model(state)
        if step in [1, 10, 30, 60, 100, 150, 200]:
            rgba = state[0, :4].clamp(0, 1).permute(1, 2, 0).numpy()
            snapshots.append((step, rgba))

fig, axes = plt.subplots(1, len(snapshots), figsize=(2.5 * len(snapshots), 2.5))
for ax, (step, img) in zip(axes, snapshots):
    ax.imshow(img)
    ax.set_title(f'step {step}', fontsize=9)
    ax.axis('off')
plt.suptitle('growth from seed', fontsize=12)
plt.tight_layout()
plt.show()

## file 2: `train.py`

this file exists because a clash game is useless if the two organisms are not real organisms first.

so the training loop had to do a few specific jobs:

- load a target image and put it on the same kind of grid the nca uses
- keep a pool of partially grown states instead of starting from scratch every time
- occasionally reinsert a fresh seed so the model still learns how to grow from nothing
- save checkpoints with enough metadata that the game can decide whether a checkpoint is usable
- later on, inject random damage during training because the clash world contains holes and disruptions

this file started simple and then got more serious once the first clash runs showed how easily ncas drift outside their comfort zone.

```python
# --- the core training loop (simplified) ---

for step in range(1, steps + 1):
    # grab a batch from the pool, always include one fresh seed
    batch_indices = random.sample(range(len(pool)), batch_size)
    batch = pool[batch_indices].clone()
    batch[0] = make_seed(...)  # one clean start per batch

    # optional: damage some samples so the model learns to recover
    batch = damage_batch(batch, probability=0.5)

    # forward: run the nca for a random number of steps
    rollout = random.randint(48, 96)
    state = batch
    for _ in range(rollout):
        state = model(state)

    # loss: just mse on rgba against the target
    loss = F.mse_loss(state[:, :4], target)

    # backward + update
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # write survivors back to the pool
    pool[batch_indices] = state.detach()
```

the full training script is ~220 lines — most of it is checkpointing, logging, and argparse.

## target loading

one detail i cared about here was matching the training canvas and the runtime canvas.

this is why targets are:

- resized to 40x40
- pasted onto a 48x48 transparent canvas

that gives the organism room to grow inward and outward around the shape, while still keeping the target compact. if the target touched the edges directly, the learned dynamics would be much touchier.

In [ ]:
import matplotlib.pyplot as plt
from train import load_target
from PIL import Image

target = load_target('targets/01_heart.png', device='cpu')
print('target tensor shape:', tuple(target.shape))
print('rgba min/max:', float(target.min()), float(target.max()))
print('nonzero alpha cells:', int((target[:, 3] > 0).sum()))

img = Image.open('targets/01_heart.png')
plt.figure(figsize=(2, 2))
plt.imshow(img)
plt.axis('off')
plt.title('01_heart target')
plt.show()

## why the pool matters

without a pool, training would mostly teach one thing: how to grow from a clean seed.

that is not enough.

i wanted the model to also learn how to continue from awkward partial states. the pool helps with that because every batch is sampled from a bag of already-grown organisms at different stages.

so the model is constantly asked to fix, continue, and stabilize states that are not identical.

that makes the final dynamics much more robust.

## why there is always one fresh seed in each batch

this is a small but important bias.

if i only trained from pooled states, the model could slowly forget how to bootstrap itself from a single live pixel. it could become good at "keep going" but bad at "start".

forcing one fresh seed into every batch keeps that skill alive.

## why i added `damage_batch`

this came later.

at first i only trained plain growth. then the moment i started thinking about clash and mouse-made craters, it was obvious that the runtime world would include missing chunks and weird wounds.

so i added damage during training.

for some fraction of the batch, i punch out a circular hole before the rollout. this forces the nca to solve not just "grow toward target" but also "repair a broken organism and still end up looking like the target".

that was one of the more practical changes in the whole project.

## why checkpoint metadata matters

this one was learned the hard way.

early on, i ran tiny smoke-training jobs just to verify that the code executed. those low-step weights were good enough for a unit test, but terrible as real organisms.

then the game loaded them because a `.pt` file existed, and the result was basically white blobs and nonsense.

so i changed the checkpoints to save `train_steps`. that let the game say:

- if this checkpoint is undertrained, do not trust it
- if bootstrap training is requested, refresh weak weights instead of blindly loading them

this is the kind of detail that feels boring until it saves you from staring at broken output for an hour.

## file 3: `clash.py`

this file is where the project stopped being a clean training exercise and became an engineering problem.

a single nca growing alone is conceptually simple. two ncas sharing a board is where all the strange edge cases start.

i ended up rewriting the clash logic more than once, because the first obvious version was wrong in a deep way.

```python
# --- separate hidden states: the key insight ---

# each organism gets its own full state tensor
state_a = make_seed(...)  # [1, 16, H, W] — organism a's private world
state_b = make_seed(...)  # [1, 16, H, W] — organism b's private world
owner = torch.zeros(1, 1, H, W)  # shared ownership grid

# --- one clash step ---

def clash_step(state_a, state_b, owner, model_a, model_b):
    state_a = model_a(state_a)
    state_b = model_b(state_b)

    # check who is alive (alpha > threshold)
    alive_a = (state_a[:, 3:4] > 0.1).float()
    alive_b = (state_b[:, 3:4] > 0.1).float()

    # claim territory: first one to grow into empty space owns it
    unclaimed = (owner == 0).float()
    owner = owner + alive_a * unclaimed * 1  # a claims
    unclaimed = (owner == 0).float()
    owner = owner + alive_b * unclaimed * 2  # b claims

    # support pruning: kill cells that wander into enemy territory
    state_a = state_a * (owner != 2).float()
    state_b = state_b * (owner != 1).float()

    return state_a, state_b, owner
```

the full file is ~290 lines — rendering, pygame loop, argparse, bootstrap logic.

## the first bad idea: one shared latent grid

my first clash version used one shared 16-channel state grid.

the thought process sounded reasonable at first:

- both models look at the same board
- both propose updates
- ownership decides whose update gets applied per cell

but that sneaks in a fatal assumption.

it assumes model a can safely read hidden channels that were produced by model b, and vice versa.

that is false.

each nca was trained on its own private hidden-state distribution. once i let one model read the other model's latent channels, both were immediately out of distribution. they started producing ugly noise, spreading into weird places, and corrupting their own future decisions.

so the correct fix was not "tune the alpha threshold" or "train more". the correct fix was to stop sharing the latent state.

## the real clash model: separate hidden states, shared territory

the current clash design works like this:

- model a owns its own full 16-channel state grid
- model b owns its own full 16-channel state grid
- the shared thing is only the owner mask
- rendering composes visible rgba from whichever organism currently owns each cell
- conflict resolution happens at the ownership layer, not by mixing hidden states

this is a much safer design because each model always reads activations that came from itself.

that one structural change fixed the nastiest failure mode in the project.

In [ ]:
from clash import ensure_model, reset_world, clash_step, list_targets
import torch

device = 'cpu'
targets = list_targets()
left_model = ensure_model(targets[0], device, 0)
right_model = ensure_model(targets[1], device, 0)
state_a, state_b, owner = reset_world(48, device)

with torch.inference_mode():
    for step in range(20):
        state_a, state_b, owner = clash_step(state_a, state_b, owner, left_model, right_model)

print('owned by a:', int((owner == 1).sum()))
print('owned by b:', int((owner == 2).sum()))
print('unclaimed:', int((owner == 0).sum()))

## why the runtime grid went back to 48x48

at one point i let the clash grid default to 64x64 because it sounded more dramatic.

that was a mistake.

the organisms were trained on 48x48. giving them a larger arena changed the long-horizon dynamics and encouraged late-stage drift into regions they were never trained to stabilize on.

so the default went back to 48x48.

that is one of those boring fixes that matters more than flashy ones. a lot of "ai system looks weird at runtime" bugs are just train-test mismatch.

## why there is support pruning in the clash loop

after the separate-state fix, there was still another ugly behavior: tiny sparks and filaments could survive in odd places, then later become seeds for junk growth.

so i added a simple structural rule:

- if a claimed cell does not have enough same-owner support in its local 3x3 neighborhood, kill it

this is not a paper-derived nca rule. it is a game-side stabilizer.

i added it because petri clash is not trying to be a pure research reproduction anymore. it is trying to behave well as an interactive system.

that tradeoff matters. once something becomes a toy you can poke with the mouse, stability starts to matter as much as theoretical purity.

### let's watch a clash unfold

run two organisms head-to-head and see territory evolve.

In [ ]:
import matplotlib.pyplot as plt
import torch
from clash import ensure_model, reset_world, clash_step, list_targets
import numpy as np

device = 'cpu'
targets = list_targets()
left = ensure_model(targets[0], device, 0)
right = ensure_model(targets[1], device, 0)
state_a, state_b, owner = reset_world(48, device)

snapshots = []
with torch.inference_mode():
    for step in range(1, 201):
        state_a, state_b, owner = clash_step(state_a, state_b, owner, left, right)
        if step % 40 == 0:
            a_count = int((owner == 1).sum())
            b_count = int((owner == 2).sum())
            # composite: red=a territory, blue=b territory
            vis = np.zeros((48, 48, 3))
            own = owner[0, 0].numpy()
            vis[own == 1] = [1, 0.3, 0.3]
            vis[own == 2] = [0.3, 0.3, 1]
            snapshots.append((step, vis, a_count, b_count))

fig, axes = plt.subplots(1, len(snapshots), figsize=(3 * len(snapshots), 3))
for ax, (step, vis, a, b) in zip(axes, snapshots):
    ax.imshow(vis)
    ax.set_title(f'step {step}\na={a} b={b}', fontsize=9)
    ax.axis('off')
plt.suptitle('clash territory over time', fontsize=12)
plt.tight_layout()
plt.show()

## the pygame loop

i kept the interface intentionally bare.

there is no hud, no graphs, no side panel, no settings menu, no tutorial screen.

that was not laziness. it was a taste decision.

the whole point of the project is to stare at the board itself. everything else would just distract from the organisms.

so the pygame loop only really does five jobs:

- open a window
- load the requested left and right targets
- step the world when not paused
- translate keyboard and mouse input into world changes
- render the composed rgba image scaled up to window size

that is exactly the amount of ui the idea needs.

## controls and why they are so minimal

these are the controls:

- `space` pause or resume
- `r` reset the board with new seed locations
- `1` through `9` change the left organism
- `shift+1` through `shift+9` change the right organism
- left click punches a crater
- `esc` quits

this keeps the interaction loop tight.

i did not want any action in the project to require leaving the visual field. that is why even target switching lives on the number keys.

## the target pack

the target icons are deliberately small and simple.

this was not because the system could never learn more detailed images. it was because these shapes are enough to expose the dynamics clearly:

- heart shows whether rounded mass can hold together
- star shows whether sharp protrusions survive
- the others give variety without requiring a huge image dataset

for a project like this, the first job of the targets is to reveal behavior, not to impress with complexity.

### the target gallery

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

target_dir = Path('targets')
pngs = sorted(target_dir.glob('*.png'))

fig, axes = plt.subplots(1, len(pngs), figsize=(1.8 * len(pngs), 2))
if len(pngs) == 1:
    axes = [axes]
for ax, p in zip(axes, pngs):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem, fontsize=8)
    ax.axis('off')
plt.suptitle('all training targets', fontsize=11)
plt.tight_layout()
plt.show()

## the sequence of bugs and fixes

this is the part i especially wanted to document, because it explains why the code looks the way it looks now.

### phase 1: get a real nca running

i built `nca.py` and `train.py`, verified that a seed could grow, and made sure checkpoints saved correctly.

### phase 2: get a playable clash loop running

i built a simple shared-grid clash loop in pygame. it technically ran, but the results were ugly.

### phase 3: fix the fake weights problem

i noticed that tiny smoke-test checkpoints were being loaded like they were real models. i added `train_steps` to checkpoints and made the loader refresh weak weights.

### phase 4: train the first real models

i trained the heart and star to 3000 steps and pushed those weights.

### phase 5: find the real clash bug

i realized the shared latent state was poisoning both models, because each one was reading hidden channels it was never trained to interpret.

### phase 6: separate the hidden states

i gave each organism its own private 16-channel state and shared only ownership plus rendering.

### phase 7: reduce late drift

i put the runtime grid back to 48x48 and added neighborhood-support pruning so tiny sparks would not become future garbage colonies.

that history is the reason the final code is not the shortest possible version. it is the version that survived contact with actual behavior.

## what i would improve next

if i kept pushing this project, these are the next things i would try.

### 1. train with clash-aware perturbations

right now the models are trained for solo growth plus crater damage. they are not explicitly trained against another live organism.

one next step would be to inject adversarial occupancy masks or partial territorial blocking during training.

### 2. separate visible state and combat confidence

right now alpha is doing two jobs:

- visible opacity
- conflict confidence

that is convenient, but maybe too entangled. a dedicated combat channel might make frontier decisions cleaner.

### 3. use living-neighbor constraints during training too

support pruning currently exists in the clash runtime. maybe a similar inductive bias should exist during training so the models learn less spark-prone dynamics naturally.

### 4. add reproducible evaluation scripts

i would like a small benchmark that measures:

- target reconstruction loss over long rollouts
- recovery after crater damage
- occupied area drift after 200 or 400 steps
- territorial stability in a two-model clash

that would make iteration much less subjective.

## how to run the project now

from the repo root:

```bash
conda activate petri-clash
python clash.py --bootstrap-steps 0
```

for training a target explicitly:

```bash
python train.py --target targets/01_heart.png --steps 3000
```

for testing the same target on both sides:

```bash
python clash.py --left 1 --right 1 --bootstrap-steps 0
```

that last one is useful when you want to see whether a weird behavior is coming from the clash logic or from one weak model.

## final thought

what i like about this project is that it only looks simple from far away.

"two little growing things fight on a board" sounds tiny. but the moment you actually implement it, you run straight into the hard parts of dynamical systems:

- hidden state distributions matter
- train-time and test-time geometry have to match
- local mistakes can become future seeds
- a model that works alone can fail badly in interaction

so the final implementation is really a sequence of guardrails around a fragile but beautiful core idea.

that is the real story of how this repo got built.